# Sprint 4 — Team & Production
### Prediction logging, drift monitoring, Docker, cloud deployment, CI

Sprint 3 put the model behind an API. Sprint 4 makes that API something a team can
run in production: every prediction is logged, drift is watched against the training
cohort, the service ships as one Docker image that deploys to Render, Railway or AWS,
and every pull request is tested automatically before it can reach `main`.

This notebook works through each checklist item against the real app, in-process
(no server needed, except in §4 where a real one is started for the smoke test).

| # | Checklist item | Section | Code |
|---|---|---|---|
| 1 | Docker containerization for ML microservices | §3 | `serving/Dockerfile`, `docker-compose.yml`, `serving/scripts/export_artifacts.py` |
| 2 | Cloud deployment (Render / AWS / Railway) | §4 | `render.yaml`, `railway.json`, `deploy/aws/`, `serving/scripts/smoke_test.py` |
| 3 | Prediction logging & monitoring data drift | §1, §2 | `serving/app/monitoring.py`, `serving/scripts/drift_report.py` |
| 4 | GitHub flow & automated testing for ML pipelines | §5 | `.github/workflows/ci.yml`, `serving/tests/test_pipeline.py` |
| 5 | Final demo and technical presentation | §6 | `docs/DEMO.md` |

Logging and drift come first because the Docker and deployment sections reuse them.

In [ ]:
%run config.ipynb

In [ ]:
import os, sys, json, subprocess, tempfile, time
from pathlib import Path
import numpy as np
import pandas as pd

SERVING_DIR = (PROJECT_ROOT / ".." / "serving").resolve()
REPO_ROOT = SERVING_DIR.parent
sys.path.insert(0, str(SERVING_DIR))
sys.path.insert(0, str(SERVING_DIR / "clients"))

# Prediction logs for this walkthrough go to a throwaway folder, not serving/logs/.
LOG_DIR = Path(tempfile.mkdtemp(prefix="screening-logs-"))
os.environ.setdefault("SCREENING_ARTIFACTS_DIR", str(PROCESSED_DATA_DIR))
os.environ["SCREENING_LOG_DIR"] = str(LOG_DIR)
# Set SCREENING_MODEL_NAME=hashing to run offline with a stand-in encoder (scores not meaningful).

import logging
from fastapi.testclient import TestClient
from app.main import create_app
from app.settings import get_settings
from screening_client import ScreeningClient
logging.getLogger("screening.api").setLevel(logging.WARNING)   # keep per-request logs out of the notebook

settings = get_settings()
app = create_app(settings)
client = TestClient(app, raise_server_exceptions=False)
client.__enter__()                                   # startup: model, embeddings, drift reference
api = ScreeningClient(http=client, max_batch_size=settings.max_batch_size)
info = client.get("/v1/model").json()
print("model:", info["model"], "| api", info["api_version"], "| build", info["build"])
print("prediction log dir:", info["monitoring"]["prediction_log_dir"])
info["monitoring"]["drift_reference"]

## 1. Prediction logging
Every resume the API scores (or rejects) becomes one JSON line in
`predictions-YYYY-MM-DD.jsonl`. Each line holds the request id, endpoint, model,
build, status, best category, similarity, latency, and a 16-character hash and the
length of the cleaned text.

**What is not logged:** the resume text. Resumes are personal data, and the hash is
enough to see that the same resume came back twice. The length is enough for drift
checks. Nothing in the log can be turned back into the resume.

In [ ]:
resumes_clean = pd.read_csv(PROCESSED_RESUMES_CSV)
sample = resumes_clean.sample(6, random_state=RANDOM_SEED)

api.match(sample.clean_text.iloc[0])                                            # single
api.match_batch([{"id": f"cand-{i}", "text": t} for i, t in enumerate(sample.clean_text[1:4])]
                + [{"id": "cand-empty", "text": ""}])                           # batch, one bad item
api.screen({"category": "ACCOUNTANT"},
           [{"id": "a", "text": sample.clean_text.iloc[4]}, {"id": "b", "text": "hi"}])   # screening

recent = client.get("/v1/monitoring/predictions", params={"limit": 20}).json()
cols = ["endpoint", "item_id", "status", "best_category", "job", "rank", "similarity",
        "text_chars", "text_sha", "reject_code", "latency_ms", "request_id"]
pd.DataFrame(recent["records"]).reindex(columns=cols)

In [ ]:
log_file = next(LOG_DIR.glob("predictions-*.jsonl"))
lines = log_file.read_text().splitlines()
print(f"{log_file.name}: {len(lines)} lines\n")
print(json.dumps(json.loads(lines[0]), indent=2))

# Prove it: no resume text in the log file.
raw = log_file.read_text()
leaks = [t[:60] for t in sample.clean_text if t[:60] in raw]
print("\nresume text found in log:", leaks or "none")

Logging is designed so it can never break a prediction. It is a single `write()` per
request under a lock, any failure is caught and logged, and a read-only disk drops to
memory-only logging instead of crashing. On hosts whose disk is wiped on every deploy
(Render, Railway), `SCREENING_LOG_PREDICTIONS_STDOUT=true` also sends each record to
stdout, where the platform keeps it. In Docker, `docker-compose.yml` mounts a volume
on `/srv/logs`.

## 2. Monitoring data drift
In production there are no labels: nobody tells the API who got hired, so accuracy
can't be measured live. What we can watch is the **inputs** and the **model's
confidence**, which change first when the world changes.

The reference is the 2,481-resume training cohort, summarised once in
`reference_profile.json` (exported with the artifacts). Recent traffic is compared
with it on six checks:

| check | question it answers |
|---|---|
| `similarity_psi` | Do resumes still look as much like our jobs as the cohort did? |
| `low_confidence_rate_delta` | Are more resumes falling under the 0.35 floor (out-of-scope)? |
| `category_psi` | Has the mix of predicted categories changed? |
| `text_length_psi` | Has the text reaching the API changed shape (new PDF parser, form field)? |
| `rejection_rate` | Is a client sending empty or garbled text? |
| `embedding_centroid_cosine` | Has the average *meaning* of the resumes moved? |

**PSI** (Population Stability Index) compares two histograms. The usual rule is under
0.10 stable, 0.10–0.25 moderate, over 0.25 significant. Those cut-offs assume large
samples, though. With 100 resumes and 24 categories, random noise alone produces a
PSI of about 0.2. So each check raises its cut-off to the level that noise exceeds
only 1% of the time. The `warn_at` and `alert_at` columns below show the cut-off used.

In [ ]:
profile = app.state.service.reference
print(f"reference: {profile.n} resumes, model={profile.model}, source={profile.source}")
print(f"cohort low-confidence rate: {profile.low_confidence_rate:.1%} (floor {profile.similarity_floor})")
pd.DataFrame({"best-similarity decile edge": profile.similarity_edges[1:-1],
              "text-length decile edge": profile.length_edges[1:-1]}).T.round(3)

### 2a. Normal traffic → no drift
Send 300 cohort resumes the server hasn't seen in this session. These look like
the training data, so every check should be `ok`.

In [ ]:
def drift_table():
    rep = client.get("/v1/monitoring/drift").json()
    print("overall:", rep["status"].upper(), "|", rep["window"]["records"], "records in window")
    return pd.DataFrame(rep["checks"])[["name", "status", "value", "warn_at", "alert_at", "samples", "detail"]]

def fresh_server():
    # A new app instance = an empty monitoring window (the log file keeps everything).
    global app, client, api
    client.__exit__(None, None, None)
    app = create_app(settings)
    client = TestClient(app, raise_server_exceptions=False); client.__enter__()
    api = ScreeningClient(http=client, max_batch_size=settings.max_batch_size)

fresh_server()
normal = resumes_clean.drop(sample.index).sample(300, random_state=1)
api.match_batch([{"id": str(i), "text": t} for i, t in zip(normal[COL_ID], normal.clean_text)])
drift_table()

### 2b. Shifted traffic → alerts
Now simulate a realistic production shift: a hiring campaign for the three roles the
model handles worst in Sprint 2 (ARTS, APPAREL, AUTOMOBILE). We also add a broken
upload path in a partner app that sends only the first 400 characters of each resume,
plus some empty files.

In [ ]:
fresh_server()
campaign = resumes_clean[resumes_clean[COL_CATEGORY].isin(["ARTS", "APPAREL", "AUTOMOBILE"])].sample(200, random_state=2)
items = [{"id": f"c{i}", "text": t} for i, t in zip(campaign[COL_ID], campaign.clean_text)]
items += [{"id": f"trunc{i}", "text": t[:400]} for i, t in enumerate(normal.clean_text[:80])]
items += [{"id": f"blank{i}", "text": ""} for i in range(30)]
api.match_batch(items)
drift_table()

The alerts tell you where to look:
* `category_psi` / `similarity_psi`: the resumes are a different population, and the model is less sure about them.
* `text_length_psi`: something upstream is cutting the text short.
* `rejection_rate`: a client is sending empty documents.

The same report is available as Prometheus gauges at `GET /metrics`
(`screening_drift_status{check=...}`: 0 ok, 1 warning, 2 alert), so Grafana or any
other alerting tool can watch it.

In [ ]:
metrics = client.get("/metrics").text
print("\n".join(l for l in metrics.splitlines()
                if l.startswith(("screening_drift_status", "screening_predictions_total", "screening_http_requests_total"))))

### 2c. Offline drift report over any period
The endpoint covers the server's in-memory window (the last 1,000 items by default).
For a weekly review, `drift_report.py` reads the JSONL files directly. `--fail-on alert`
returns exit code 2, so a cron job or scheduled CI run can page someone.

In [ ]:
# Use the exact reference the server is running with (exported with the artifacts,
# or rebuilt at startup if you're on the offline stand-in encoder).
profile_path = LOG_DIR / "reference_profile.json"
profile_path.write_text(app.state.service.reference.to_json())
out = subprocess.run([sys.executable, str(SERVING_DIR / "scripts" / "drift_report.py"),
                      "--logs", str(LOG_DIR), "--profile", str(profile_path), "--fail-on", "alert"],
                     capture_output=True, text=True)
print(out.stdout or out.stderr)
print("exit code:", out.returncode)

## 3. Docker containerization for ML microservices
`serving/Dockerfile` builds one self-contained image:

| Decision | Why |
|---|---|
| CPU-only torch wheel | About 1 GB image instead of about 5 GB with CUDA. Inference here is CPU-bound anyway |
| Model weights baked in at build time, `HF_HUB_OFFLINE=1` | Containers start without internet, and every replica runs byte-identical weights |
| Dependencies installed before code is copied | Editing a `.py` file rebuilds in seconds (cached layers) |
| Committed artifact bundle copied to `/data` | A fresh `git clone` builds. No notebooks or Kaggle download are needed in CI or on Render |
| Non-root `app` user, only `/srv/logs` writable | Standard container hardening |
| `--port ${PORT}` | Render, Railway and ECS inject the port. The same image runs everywhere |
| `HEALTHCHECK` on `/ready` | Docker and the platforms only route traffic once the model is loaded |
| 1 uvicorn worker per container | Each worker holds its own model copy (~400 MB), so scale with containers |

The bundle is the one new data artefact. `data_pipeline/data/` is gitignored, so Sprint 3's
image could only be built on a laptop that had run the notebooks.
`export_artifacts.py` writes the ~6 MB subset the server needs, plus a sha256
manifest that CI verifies:

In [ ]:
manifest = json.loads((SERVING_DIR / "artifacts" / "manifest.json").read_text())
print("created:", manifest["created_at"], "| model:", manifest["model"], "| resumes:", manifest["resumes"])
pd.DataFrame([{"file": k, "MB": round(v["bytes"] / 1e6, 2), "sha256": v["sha256"][:12]}
              for k, v in manifest["files"].items()])

After re-running the Sprint 1 notebooks, refresh the bundle, then build and run:
```bash
python serving/scripts/export_artifacts.py          # from the repo root
docker compose up --build                            # → http://localhost:8000/docs
```

## 4. Cloud deployment (Render / AWS / Railway)
All three run the **same image**. Only the platform config differs:

| Platform | File | How it deploys | Notes |
|---|---|---|---|
| Render | `render.yaml` (Blueprint) | `autoDeployTrigger: checksPass` deploys each commit on `main` once CI is green | 1 CPU / 2 GB plan. A generated API key. Prediction records go to stdout because the disk is ephemeral |
| Railway | `railway.json` | GitHub integration with "Wait for CI" | Same health check (`/ready`) |
| AWS | `deploy/aws/deploy_ecs_express.sh` | Push to ECR, then **ECS Express Mode** (Fargate + load balancer + HTTPS URL) | App Runner stopped accepting new customers on 30 Apr 2026. AWS recommends ECS Express Mode instead |

Whatever the platform, deployment ends with the same check:
`smoke_test.py` hits the live URL and verifies the model loaded, the saved embeddings
were reused, auth works, a CPA resume lands in ACCOUNTANT or FINANCE, and monitoring is
recording. Here it runs against a real uvicorn server started from this notebook:

In [ ]:
env = {**os.environ, "SCREENING_LOG_DIR": str(LOG_DIR / "smoke"), "SCREENING_API_KEY": "demo-key"}
server = subprocess.Popen([sys.executable, "-m", "uvicorn", "app.main:app", "--port", "8765"],
                          cwd=SERVING_DIR, env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
try:
    out = subprocess.run([sys.executable, str(SERVING_DIR / "scripts" / "smoke_test.py"),
                          "--url", "http://127.0.0.1:8765", "--api-key", "demo-key", "--wait", "180"],
                         capture_output=True, text=True)
    print(out.stdout or out.stderr)
finally:
    server.terminate(); server.wait()

## 5. GitHub flow & automated testing for ML pipelines
**Flow:** `main` is always deployable. Work happens on a short-lived branch, gets a pull
request, needs green CI and one review, and is squash-merged. Merging is deploying.
See `docs/CONTRIBUTING.md`.

```
feature/drift-alerts ──PR──► CI: lint · test (3.11, 3.12) · docker build + smoke ──► review ──► squash-merge to main
                                                                                                   │
                                          CI again on main ──► image pushed to GHCR ──► Render deploys (checksPass)
```

The test suite now covers the ML pipeline as well as the API:

| File | Guards against |
|---|---|
| `test_api.py`, `test_clients.py` | Endpoint contracts, error codes, caching, client libraries (Sprint 3) |
| `test_monitoring.py` | PSI maths and noise-aware thresholds, log format, *no resume text in logs*, metrics |
| `test_pipeline.py` | Non-portable notebooks, committed credentials, **training/serving skew in text cleaning**, a corrupted or misaligned artifact bundle, an **accuracy drop below the Sprint 2 baseline** (quality gate), and drift checks that alarm on normal traffic |
| `test_deploy.py` | Dockerfile, Render, Railway, compose and CI configs out of sync with the code, plus undocumented endpoints |

Two real problems were caught while writing these:
1. `tfidf_baseline.ipynb` still sat at the repo top level with `C:\Users\...` paths. Sprint 1's fix never made it into the archive. It now lives in `data_pipeline/` and uses `config.ipynb`.
2. The API's regex HTML-stripper cleaned 13 of the 4,968 raw Kaggle texts differently from `preprocess.ipynb`'s BeautifulSoup version. `serving/app/text.py` now uses the same call, and a test compares the two on the full dataset.

In [ ]:
clean_env = {k: v for k, v in os.environ.items() if not k.startswith("SCREENING_")}
out = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider"], cwd=SERVING_DIR, env=clean_env,
                     capture_output=True, text=True)
print(out.stdout[-1500:])

## 6. Final demo
The live demo script (≈8 minutes) is in `docs/DEMO.md`. The core of it is one recruiter
action, followed by what an operator sees:

In [ ]:
fresh_server()
applicants = [{"id": f"applicant-{i}", "text": t}
              for i, t in enumerate(resumes_clean.sample(10, random_state=7).clean_text)]
applicants += [{"id": "applicant-blank", "text": ""},
               {"id": "applicant-sax", "text": "Professional saxophonist with ten years touring internationally, "
                                               "performing jazz concerts and teaching music theory."}]
result = api.screen(job={"description": "Hiring a payroll accountant: general ledger, month-end close, "
                                        "reconciliations, tax filings and audit support."},
                    candidates=applicants, top_k=5)
print("rejected:", [c["id"] for c in result["rejected"]])
pd.DataFrame(result["ranked"])

In [ ]:
# ...and the operator's view of that same request
pd.DataFrame(client.get("/v1/monitoring/predictions", params={"endpoint": "screen"}).json()["records"]) \
  .reindex(columns=["item_id", "status", "rank", "similarity", "text_chars", "reject_code", "latency_ms", "request_id"])

In [ ]:
client.__exit__(None, None, None)
print("Done. Production run:  docker compose up --build   →  http://localhost:8000/docs")

---
## Notes
* Numbers depend on your machine and on the model. With `SCREENING_MODEL_NAME=hashing`,
  the drift reference is rebuilt for the stand-in encoder, so the checks still behave
  sensibly but the similarity values mean nothing.
* The drift checks measure *change*, not *correctness*. A clean drift report doesn't
  prove the rankings are right. That still needs labelled outcomes (see "Next steps" in
  the progress report).